In [2]:
import os
from pathlib import Path
import json

from conch.open_clip_custom import create_model_from_pretrained
from conch.downstream.zeroshot_path import zero_shot_classifier, run_zeroshot

import torch
import torch.nn.functional as F
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from conch.open_clip_custom import tokenize, get_tokenizer
from tqdm import tqdm
import json
import itertools
from pathlib import Path
from PIL import Image
import csv
import re

In [3]:
model_cfg = 'conch_ViT-B-16'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

checkpoint_paths = [
    'checkpoints/conch/conch_mhist_finetuned_num_unfrozen=1.bin',
    'checkpoints/conch/conch_mhist_finetuned_num_unfrozen=2.bin',
    'checkpoints/conch/conch_mhist_finetuned_num_unfrozen=4.bin',
]

prompt_files = {
    'all_prompts': 'CONCH/prompts/crc100k_prompts_all_per_class.json',
    'pruned':      'CONCH/prompts/crc100k_prompts_pruned.json',
}

image_dir = Path("mhist_data/images")
k = 10  # top-k prompts kept per image

tokenizer = get_tokenizer()


In [4]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def expand_prompts(prompt_file):
    """Expand classnames × templates into a flat list of prompts."""
    with open(prompt_file) as f:
        data = json.load(f)["0"]
    classnames = data["classnames"]
    templates  = data["templates"]
    prompts = []
    for _, synonyms in classnames.items():
        for syn in synonyms:
            for template in templates:
                if "CLASSNAME" in template:
                    prompts.append(template.replace("CLASSNAME", syn))
    return prompts


def num_unfrozen_tag(ckpt_path):
    """Pull num_unfrozen=<n> from the filename; base/pretrained model = 0."""
    m = re.search(r'num_unfrozen=(\d+)', ckpt_path)
    return m.group(1) if m else '0'

In [5]:
# ── Pre-load prompt sets (once) ───────────────────────────────────────────────
prompt_sets = {tag: expand_prompts(path) for tag, path in prompt_files.items()}
for tag, prompts in prompt_sets.items():
    print(f"  {tag}: {len(prompts)} prompts (e.g., '{prompts[0]}')")

image_paths = sorted(image_dir.glob("*.png"))
print(f"Found {len(image_paths)} images")

  all_prompts: 902 prompts (e.g., 'adipose.')
  pruned: 560 prompts (e.g., 'background.')
Found 3152 images


In [6]:
# ── Loop: one image-encoding pass per model, then both prompt sets ────────────
for ckpt in checkpoint_paths:
    n_unfrozen = num_unfrozen_tag(ckpt)
    print(f"\n{'='*72}\nCheckpoint: {ckpt}  (num_unfrozen={n_unfrozen})\n{'='*72}")

    model, preprocess = create_model_from_pretrained(model_cfg, ckpt, device=device)
    model = model.eval()

    # Encode all images ONCE for this checkpoint, reuse across both prompt sets
    image_names = []
    img_feats = []
    with torch.no_grad():
        for img_path in tqdm(image_paths, desc="Encoding images"):
            image = Image.open(img_path).convert("RGB")
            image_tensor = preprocess(image).unsqueeze(0).to(device)
            f = model.encode_image(image_tensor)
            f = f / f.norm(dim=-1, keepdim=True)
            img_feats.append(f.cpu())
            image_names.append(img_path.name)
    img_feats = torch.cat(img_feats, dim=0)  # [N, D] on CPU

    # Now run both prompt sets against the cached image features
    for prompt_tag, prompts in prompt_sets.items():
        print(f"\n  → prompt set: {prompt_tag} ({len(prompts)} prompts)")

        tokenized = tokenize(texts=prompts, tokenizer=tokenizer).to(device)
        with torch.no_grad():
            text_feats = model.encode_text(tokenized)
            text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)
        text_feats = text_feats.cpu()

        sims = img_feats @ text_feats.T          # [N, P]
        topk_vals, topk_idxs = sims.topk(k, dim=1)

        out_name = f"mhist_with_captions_num_unfrozen={n_unfrozen}_{prompt_tag}.csv"
        with open(out_name, mode='w', newline='') as csvfile:
            writer = csv.DictWriter(
                csvfile, fieldnames=["image", "top_prompts", "top_scores"]
            )
            writer.writeheader()
            for i, name in enumerate(image_names):
                idxs = topk_idxs[i].tolist()
                vals = topk_vals[i].tolist()
                writer.writerow({
                    "image":       name,
                    "top_prompts": [prompts[j] for j in idxs],
                    "top_scores":  vals,
                })
        print(f"  Wrote {out_name}")

    del model, img_feats
    torch.cuda.empty_cache()


Checkpoint: checkpoints/conch/conch_mhist_finetuned_num_unfrozen=1.bin  (num_unfrozen=1)


Encoding images: 100%|██████████| 3152/3152 [12:58<00:00,  4.05it/s]



  → prompt set: all_prompts (902 prompts)
  Wrote mhist_with_captions_num_unfrozen=1_all_prompts.csv

  → prompt set: pruned (560 prompts)
  Wrote mhist_with_captions_num_unfrozen=1_pruned.csv

Checkpoint: checkpoints/conch/conch_mhist_finetuned_num_unfrozen=2.bin  (num_unfrozen=2)


Encoding images: 100%|██████████| 3152/3152 [14:10<00:00,  3.70it/s]



  → prompt set: all_prompts (902 prompts)
  Wrote mhist_with_captions_num_unfrozen=2_all_prompts.csv

  → prompt set: pruned (560 prompts)
  Wrote mhist_with_captions_num_unfrozen=2_pruned.csv

Checkpoint: checkpoints/conch/conch_mhist_finetuned_num_unfrozen=4.bin  (num_unfrozen=4)


Encoding images: 100%|██████████| 3152/3152 [13:06<00:00,  4.01it/s]



  → prompt set: all_prompts (902 prompts)
  Wrote mhist_with_captions_num_unfrozen=4_all_prompts.csv

  → prompt set: pruned (560 prompts)
  Wrote mhist_with_captions_num_unfrozen=4_pruned.csv
